In [2]:
%pip install scikit-learn

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/8.2 MB 3.3 MB/s eta 0:00:03
   ---------- ----------------------------- 2.1/8.2 MB 3.8 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.2 MB 4.2 MB/s eta 0:00:02
   -------------------- ------------------- 4.2/8.2 MB 4.3 MB/s eta 0:00:01
   -------------------------- ------------- 5.5/8.2 MB 4.6 MB/s eta 0:00:01
   ------------------------------- -------- 6.6/8.2 MB 4.6 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.2 MB 4.3 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.2 MB 4.2 MB/s e

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [4]:
df = pd.read_csv('../../data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Data prep

In [5]:
print(df.isnull().sum())

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [23]:

cat = df.select_dtypes(include='object').columns.tolist()
num = [c for c in df.select_dtypes(exclude='object').columns if c != 'converted']
df[cat] = df[cat].fillna('NA')
df[num] = df[num].fillna(0.0)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_18900\2210884922.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat = df.select_dtypes(include='object').columns.tolist()


In [24]:
print('categorical:', cat)

categorical: ['lead_source', 'industry', 'employment_status', 'location']


In [25]:
print('numerical:', num)

numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


### Q1

In [30]:
print(df['industry'].value_counts())

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64


### Q2

In [31]:
print(df[num].corr())

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


## Split

In [32]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)
y_train, y_val = df_train.converted.values, df_val.converted.values
df_train = df_train.drop(columns='converted')
df_val = df_val.drop(columns='converted')

### Q3 (training set only; mutual_info_score needs the target, so use y_train)

In [16]:
cat = [c for c in cat if c in df_train.columns]
for c in cat:
    print(c, round(mutual_info_score(df_train[c], y_train), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


## func

In [17]:
def fit_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_va = dv.transform(df_val[features].to_dict(orient='records'))
    m = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    m.fit(X_tr, y_train)
    return accuracy_score(y_val, m.predict(X_va))

features = cat + num

### Q4

In [18]:
base = fit_eval(features)
print(round(base, 2))

0.65


### Q5: drop each candidate feature, compare to unrounded baseline

In [19]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    acc = fit_eval([x for x in features if x != f])
    print(f, base - acc)   # smallest difference wins

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


### Q6

In [20]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(fit_eval(features, C=C), 3))  # ties -> pick smallest C

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645
